In [1]:
import pandas as pd
import os
from openai import OpenAI
from dotenv import load_dotenv
from sklearn.metrics import classification_report
import time

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [2]:
train = pd.read_csv('data/train_final.csv')
test = pd.read_csv('data/test_final.csv')

classes = train['label'].unique().tolist()
print(classes)

['Fileservice', 'Support general', 'Software', 'O365', 'Active Directory', 'Computer-Services']


In [3]:
def classify_zero_shot(ticket_text):
    prompt = f"""You are an IT helpdesk ticket classifier.
Classify the following support ticket into exactly one of these categories:
{', '.join(classes)}

Ticket: {ticket_text}

Reply with only the category name, nothing else."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()  

In [4]:
results = []
for i, row in test.iterrows():
    try:
        pred = classify_zero_shot(row['text_clean'])
        results.append(pred)
        if i % 50 == 0:
            print(f"{i}/{len(test)} done")
        time.sleep(0.1)  # rate limiting
    except Exception as e:
        print(f"Error at {i}: {e}")
        results.append("Error")

test['pred_zero_shot'] = results
print(classification_report(test['label'], test['pred_zero_shot'])) 

0/599 done
50/599 done
100/599 done
150/599 done
200/599 done
250/599 done
300/599 done
350/599 done
400/599 done
450/599 done
500/599 done
550/599 done
                   precision    recall  f1-score   support

 Active Directory       0.30      0.71      0.42        49
Computer-Services       0.42      0.63      0.50        41
      Fileservice       0.91      0.97      0.94       138
             O365       0.81      0.89      0.85        92
         Software       0.57      0.53      0.55        58
  Support general       0.80      0.43      0.55       221

         accuracy                           0.67       599
        macro avg       0.63      0.70      0.64       599
     weighted avg       0.74      0.67      0.67       599



In [ ]:
# ticket = input("Enter a support ticket: ")
# prediction = classify_zero_shot(ticket)
# print(f"Predicted category: {prediction}")

Predicted category: Computer-Services


### Few-shot

In [1]:
import pandas as pd
import os
import time
from openai import OpenAI
from dotenv import load_dotenv
from sklearn.metrics import classification_report

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

classes = ['Fileservice', 'Support general', 'O365', 'Software', 'Active Directory', 'Computer-Services']

In [2]:
few_shot_examples = {
    "Fileservice": [
        "I need access to the shared folder on the finance network drive",
        "Please grant read permission to the project files for my team"
    ],
    "Support general": [
        "My computer is running very slow and I need someone to help",
        "The monitor keeps flickering and is hard to work with"
    ],
    "O365": [
        "I cannot login to my Outlook account since this morning",
        "Teams is not loading and I have a meeting in 10 minutes"
    ],
    "Software": [
        "Adobe Acrobat crashes every time I try to open a PDF",
        "The accounting software is throwing a license error on startup"
    ],
    "Active Directory": [
        "My account has been locked out and I cannot log in to the domain",
        "Please reset my domain password I have been locked out"
    ],
    "Computer-Services": [
        "The office printer is not connecting to any computers on the floor",
        "My desktop computer will not power on this morning"
    ]
}

In [3]:
def classify_few_shot(text):
    examples_text = ""
    for label, examples in few_shot_examples.items():
        for ex in examples:
            examples_text += f"Ticket: {ex}\nCategory: {label}\n\n"

    prompt = f"""You are an IT helpdesk ticket classifier.
Here are some example tickets and their correct categories:

{examples_text}
Now classify this ticket into exactly one of these categories:
{', '.join(classes)}

Ticket: {text}

Reply with only the category name, nothing else."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()

In [5]:
test_sample = pd.read_csv('data/test_final.csv')

results_fs = []
print(f"Running few-shot GPT on {len(test_sample)} tickets...\n")
for i, row in test_sample.iterrows():
    try:
        pred = classify_few_shot(row['text_clean'])
        results_fs.append(pred)
        if i % 50 == 0:
            print(f"{i}/{len(test_sample)} done")
        time.sleep(0.1)
    except Exception as e:
        print(f"Error at {i}: {e}")
        results_fs.append("Error")

test_sample['pred_few_shot'] = results_fs

print("\n===== FEW-SHOT RESULTS =====\n")
for _, row in test_sample.iterrows():
    status = "✅" if row['label'] == row['pred_few_shot'] else "❌"
    print(f"{status}  True: {row['label']:<20} Predicted: {row['pred_few_shot']}")

correct = (test_sample['label'] == test_sample['pred_few_shot']).sum()
total = len(test_sample)
print(f"\n===== SUMMARY =====")
print(f"Correct : {correct}/{total}")
print(f"Accuracy: {correct/total*100:.1f}%")
print("\n===== CLASS REPORT =====")
print(classification_report(test_sample['label'], test_sample['pred_few_shot']))

test_sample.to_csv('results/gpt_fewshot_predictions.csv', index=False)
print("\nSaved to results/gpt_fewshot_predictions.csv")

Running few-shot GPT on 599 tickets...

0/599 done
50/599 done
100/599 done
150/599 done
200/599 done
250/599 done
300/599 done
350/599 done
400/599 done
450/599 done
500/599 done
550/599 done

===== FEW-SHOT RESULTS =====

✅  True: Support general      Predicted: Support general
✅  True: Fileservice          Predicted: Fileservice
✅  True: Software             Predicted: Software
✅  True: Fileservice          Predicted: Fileservice
❌  True: Software             Predicted: Active Directory
✅  True: Support general      Predicted: Support general
✅  True: Support general      Predicted: Support general
✅  True: Fileservice          Predicted: Fileservice
✅  True: Computer-Services    Predicted: Computer-Services
✅  True: Active Directory     Predicted: Active Directory
❌  True: Support general      Predicted: O365
✅  True: Support general      Predicted: Support general
❌  True: Support general      Predicted: Active Directory
✅  True: Support general      Predicted: Support general
✅  